In [9]:
from langchain_aws import ChatBedrockConverse
from langchain_ollama import ChatOllama
from typing import TypedDict, Literal,List,Dict,Annotated
from langgraph.graph import StateGraph,START,END,MessagesState
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage,BaseMessage,ToolMessage
import operator
from langgraph.prebuilt.tool_node import ToolNode
from langchain_core.tools import tool
import uuid
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command,interrupt



In [10]:
llm = ChatOllama(model='qwen2.5:1.5b',temperature=0.2,max_token=100)

In [11]:
class StateSchema(MessagesState):
    pass
  
    

In [30]:
def llm_call(state:StateSchema):
    messages = state['messages']
    res = llm.invoke(messages)
    return {'messages': [res]}



In [31]:
graph = StateGraph(StateSchema)
graph.add_node('llm_call_node',llm_call)

graph.add_edge(START,'llm_call_node')
graph.add_edge('llm_call_node',END)

conn = sqlite3.connect("database/data.db",check_same_thread=False)
workflow = graph.compile(checkpointer=SqliteSaver(conn=conn))

In [32]:
print(workflow.get_graph().draw_ascii())

  +-----------+    
  | __start__ |    
  +-----------+    
        *          
        *          
        *          
+---------------+  
| llm_call_node |  
+---------------+  
        *          
        *          
        *          
   +---------+     
   | __end__ |     
   +---------+     


In [35]:
thread_id = uuid.uuid4()
config = {'configurable':{'thread_id':thread_id}}
res = workflow.invoke({'messages':[HumanMessage(content="ventilator is used for what in 1 line ?")]},config=config)
res

{'messages': [HumanMessage(content='ventilator is used for what in 1 line ?', additional_kwargs={}, response_metadata={}, id='4236fc1f-b39a-4bbd-9c20-e6f6c7101dc0'),
  AIMessage(content='Ventilators are used to assist or support breathing in patients who cannot breathe on their own due to illness, injury, or other medical conditions.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-09-20T09:26:06.197363759Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1310885551, 'load_duration': 201975897, 'prompt_eval_count': 41, 'prompt_eval_duration': 35620000, 'eval_count': 30, 'eval_duration': 1054586000, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a0be23-1355-7a63-9a3a-f40eb65b56e2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 30, 'total_tokens': 71})]}

In [36]:
res = workflow.invoke({'messages':[HumanMessage(content="what can be the approx cost of it ?")]},config=config)
res

{'messages': [HumanMessage(content='ventilator is used for what in 1 line ?', additional_kwargs={}, response_metadata={}, id='4236fc1f-b39a-4bbd-9c20-e6f6c7101dc0'),
  AIMessage(content='Ventilators are used to assist or support breathing in patients who cannot breathe on their own due to illness, injury, or other medical conditions.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-09-20T09:26:06.197363759Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1310885551, 'load_duration': 201975897, 'prompt_eval_count': 41, 'prompt_eval_duration': 35620000, 'eval_count': 30, 'eval_duration': 1054586000, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a0be23-1355-7a63-9a3a-f40eb65b56e2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 30, 'total_tokens': 71}),
  HumanMessage(content='what can be the approx cost of it ?', additional_kwargs={}, response_metadat

In [17]:
chat_history = list(workflow.get_state_history(config=config))
for msg in chat_history[::-1]:
    print(msg)

StateSnapshot(values={'messages': []}, next=('__start__',), config={'configurable': {'thread_id': '48928dec-4546-4890-ab88-ce21202949e4', 'checkpoint_ns': '', 'checkpoint_id': '1f1b4d49-1e31-693f-bfff-22f34ea9c793'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-09-20T09:20:49.884177+00:00', parent_config=None, tasks=(PregelTask(id='95806db6-f44d-7c2f-1157-0e3f37056484', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'messages': [HumanMessage(content='ventilator is used for what ?', additional_kwargs={}, response_metadata={}, id='20106382-e7c8-4a51-9165-1e58c65fe67b')]}),), interrupts=())
StateSnapshot(values={'messages': [HumanMessage(content='ventilator is used for what ?', additional_kwargs={}, response_metadata={}, id='20106382-e7c8-4a51-9165-1e58c65fe67b')]}, next=('llm_call_node',), config={'configurable': {'thread_id': '48928dec-4546-4890-ab88-ce21202949e4', 'checkpoint_ns': '', 'checkpoint_id': 